In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
create schema if not exists retailanalytics.gold

In [0]:
%sql

CREATE TABLE IF NOT EXISTS retailanalytics.gold.dim_products
(
    ProductID INT,
    ProductName STRING,
    Category STRING,
    SubCategory STRING,
    Brand STRING,
    CostPrice DECIMAL(10,2)
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.gold.dim_customers
(
    CustomerID INT,
    FirstName STRING,
    LastName STRING,
    Email STRING,
    Phone STRING,
    City STRING,
    State STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.gold.fact_sales
(
    OrderID BIGINT,
    CustomerID INT,
    ProductID INT,
    OrderDate DATE,
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    SalesAmount DECIMAL(18,2),
    ProfitAmount DECIMAL(18,2),
    StoreCode STRING
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.gold.fact_exchange_rates
(
    BaseCurrency STRING,
    TargetCurrency STRING,
    ExchangeRate DOUBLE,
    RateDate DATE
)
USING DELTA;

CREATE TABLE IF NOT EXISTS retailanalytics.gold.fact_sales_summary
(
    OrderDate DATE,
    StoreCode STRING,
    TotalOrders BIGINT,
    TotalQuantity BIGINT,
    TotalSales DECIMAL(18,2)
)
USING DELTA;

In [0]:
gold_configs = [
    {"source": "products","target": "dim_products"},
    {"source": "customers","target": "dim_customers"}
]

In [0]:
for config in gold_configs:

    source = config["source"]
    target = config["target"]
    df = spark.table(f"retailanalytics.silver.{source}")

    if target == "dim_products":
        df=(df.select("ProductID","ProductName","Category","SubCategory","Brand","CostPrice"))

    elif target == "dim_customers":
        df=(df.select("CustomerID","FirstName","LastName","Email","Phone","City","State"))

    (df.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(f"retailanalytics.gold.{target}"))

    print(f"Loaded : {target}")

Loaded : dim_products
Loaded : dim_customers


In [0]:
products_df = spark.table("retailanalytics.silver.products")
orders_df = spark.table("retailanalytics.silver.orders")

fact_sales = (
    orders_df.join(products_df.select("ProductID","CostPrice"),"ProductID","left")
    .withColumn("SalesAmount",col("Quantity") * col("UnitPrice"))
    .withColumn("ProfitAmount",col("Quantity") *(col("UnitPrice")-col("CostPrice")))
    .select("OrderID","CustomerID","ProductID","OrderDate","Quantity","UnitPrice",
    "SalesAmount","ProfitAmount","StoreCode")
)

(fact_sales.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("retailanalytics.gold.fact_sales"))

print("Loaded : fact_sales")

Loaded : fact_sales


In [0]:
fact_exchange = (spark.table("retailanalytics.silver.exchange_rates")
    .select("BaseCurrency","TargetCurrency","ExchangeRate","RateDate"))

(fact_exchange.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("retailanalytics.gold.fact_exchange_rates"))

print("Loaded : fact_exchange_rates")

Loaded : fact_exchange_rates


In [0]:
sales_summary = (fact_sales.groupBy("OrderDate","StoreCode")
.agg(count("OrderID").alias("TotalOrders"),sum("Quantity").alias("TotalQuantity"),
round(sum("SalesAmount"),2).alias("TotalSales")))

(sales_summary.write.format("delta").mode("overwrite").option("overwriteSchema","true")
.saveAsTable("retailanalytics.gold.fact_sales_summary"))

print("Loaded : fact_sales_summary")

Loaded : fact_sales_summary


In [0]:
gold_tables = ["dim_products","dim_customers","fact_sales","fact_exchange_rates","fact_sales_summary"]

for table in gold_tables:
    print("\n")
    print("=" * 50)
    print(table.upper())
    print("=" * 50)
    display(spark.table(f"retailanalytics.gold.{table}").limit(5))

    print("Count :",spark.table(f"retailanalytics.gold.{table}").count())



DIM_PRODUCTS


ProductID,ProductName,Category,SubCategory,Brand,CostPrice
1,Laptop_1,Electronics,Laptop,Sony,252.44
2,Jacket_2,Fashion,Jacket,Samsung,679.93
3,Tablet_3,Electronics,Tablet,Nike,41.46
4,Laptop_4,Electronics,Laptop,Apple,510.30
5,Tablet_5,Electronics,Tablet,Apple,718.86


Count : 62


DIM_CUSTOMERS


CustomerID,FirstName,LastName,Email,Phone,City,State
9,Karthik,Patel,customer9@gmail.com,9696355850,Bangalore,Maharashtra
1,Arjun,Patel,customer1@gmail.com,9927622808,Mumbai,Maharashtra
2,Arjun,Iyer,customer2@gmail.com,9885934963,Mumbai,Maharashtra
3,Anjali,Patel,customer3@gmail.com,9959015427,Delhi,Delhi
4,Rahul,Singh,customer4@gmail.com,9186064385,Chennai,Tamil Nadu


Count : 1000


FACT_SALES


...

Count : 5000


FACT_EXCHANGE_RATES


BaseCurrency,TargetCurrency,ExchangeRate,RateDate
USD,INR,86.54,2026-06-25
USD,EUR,0.91,2026-06-25
USD,GBP,0.78,2026-06-25
USD,AED,3.67,2026-06-25


Count : 4


FACT_SALES_SUMMARY


OrderDate,StoreCode,TotalOrders,TotalQuantity,TotalSales
2025-03-13,CHN001,6,16,45298.90
2025-05-12,BLR001,7,25,66785.47
2025-05-08,CHN001,7,17,53936.02
2025-06-04,DEL001,8,28,94776.21
2025-04-27,HYD001,3,10,27532.87


Count : 900
